In [6]:

from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(
    model="gapgpt-qwen-3.5",
    temperature=0.3,
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_KEY")
)

INPUT_FILE = "../inputs/yaskawa-l1000a.txt"

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from libAgent.markdownSplitter import markdownTextSplitter
chunks = markdownTextSplitter(INPUT_FILE)

print(len(chunks))

1353


In [9]:
print( llm.invoke('hello') )

content='Hello! How can I help you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3.6-35B-A3B-FP8', 'system_fingerprint': None, 'id': 'chatcmpl-c6c92b239048a3f7d9611163996f68a7', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f06d4-95dc-77b0-8286-630d5e2c59c6-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23, 'input_token_details': {}, 'output_token_details': {}}


#Extractor

In [19]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(enc.encode(text))

MIN_TOKENS = 1000

merged_chunks = []

i = 0
while i < len(chunks):
    current = chunks[i].page_content
    tokens = count_tokens( current )

    while tokens < MIN_TOKENS and i + 1 < len(chunks):
        i += 1
        current += "\n\n" + chunks[i].page_content
        tokens = count_tokens(current)

    merged_chunks.append(current)
    i += 1

print( len(merged_chunks) )

225


In [20]:
print( merged_chunks[0])

<!-- image -->

Models: 200 V Class: 1.5 to 110 kW  
Type: CIMR-LB   
A                   , CIMR-LT  A  
400 V Class: 1.5 to 315 kW  
To properly use the product, read this manual thoroughly and retain for easy reference, inspection, and maintenance. Ensure the end user receives this manual.  
<!-- image -->  
Receiving  
Mechanical Installation  
Electrical Installation  
Start-Up Programming &amp; Operation  
Parameter Details  
Troubleshooting  
Periodic Inspection &amp; Maintenance  
Peripheral Devices &amp; Options  
Specifications  
Parameter List  
MEMOBUS/Modbus Communications  
Standards Compliance  
Quick Reference Sheet  
1  
1  
2  
2  
3  
3  
4  
4  
5  
5  
- 6 6  
7  
7  
8  
8  
A  
A  
B  
B  
C  
C  
<!-- image -->  
E  
E

All rights reserved.  No part of this publication may be reproduced, stored in a retrieval system, or transmitted, in any form, or by any means, mechanical, electronic, photocopying, recording, or otherwise, without the prior written permission of

In [14]:
from pydantic import BaseModel, Field
from typing import List 

class Extracted(BaseModel):

    main_topics: List[str] = Field(
        default_factory=list,
        description="High-level topics explicitly present in the text"
    )

    subtopics: List[str] = Field(
        default_factory=list,
        description="Detailed concepts under main topics"
    )

    summary : str = Field(
        default_factory=str,
        description="summary of content"
    )
    # entities: List[str] = Field(
    #     default_factory=list,
    #     description="Named entities like devices, error codes, APIs"
    # )

    # keywords: List[str] = Field(
    #     default_factory=list,
    #     description="Searchable terms for retrieval"
    # )

llmExtranct = llm.with_structured_output(Extracted)
from langchain_core.prompts import ChatPromptTemplate

In [ ]:


prompt = ChatPromptTemplate.from_messages([
        (
        "system",
        """
        You are a knowledge extraction system.

        Analyze the provided document chunk and extract:

        1. Main Topics
        - High-level subjects discussed in the chunk.
        - Use concise canonical names.
        - Maximum 5 items.

        2. Subtopics
        - More specific concepts under the main topics.
        - Maximum 10 items.

        3. summary of document chunck 
        - max 4 line

        Rules:
        - Do not invent information.
        - Extract only information explicitly present in the text.
        - Remove duplicates.
        - Use canonical names whenever possible.
        - Return JSON only.

        Output schema:
        
        "main_topics": [],
        "subtopics": [],
        "summary:''

        """
    ),
    (
        "user",
        """
        Document Chunk:

        {chunk}
        """
    ),
])

chain  = prompt | llmExtranct


extraxted = []
for i in range(0,len(merged_chunks)):
    extraxted.append( chain.invoke({'chunk':merged_chunks[i]}).model_dump_json() )
    print(f'extract {i} / {len(merged_chunks)}')

# print( chain.invoke({'chunk':merged_chunks[0]}) ) 

extract 0 / 225
extract 1 / 225
extract 2 / 225
extract 3 / 225
extract 4 / 225
extract 5 / 225
extract 6 / 225
extract 7 / 225
extract 8 / 225
extract 9 / 225
extract 10 / 225
extract 11 / 225
extract 12 / 225
extract 13 / 225
extract 14 / 225
extract 15 / 225
extract 16 / 225
extract 17 / 225
extract 18 / 225
extract 19 / 225
extract 20 / 225
extract 21 / 225
extract 22 / 225
extract 23 / 225
extract 24 / 225
extract 25 / 225
extract 26 / 225
extract 27 / 225
extract 28 / 225
extract 29 / 225
extract 30 / 225
extract 31 / 225
extract 32 / 225
extract 33 / 225
extract 34 / 225
extract 35 / 225
extract 36 / 225
extract 37 / 225
extract 38 / 225
extract 39 / 225
extract 40 / 225
extract 41 / 225
extract 42 / 225
extract 43 / 225
extract 44 / 225
extract 45 / 225
extract 46 / 225
extract 47 / 225
extract 48 / 225
extract 49 / 225
extract 50 / 225
extract 51 / 225
extract 52 / 225
extract 53 / 225
extract 54 / 225
extract 55 / 225
extract 56 / 225
extract 57 / 225
extract 58 / 225
extract

In [23]:
# extraxtedJson = [ item.json() for item in extraxted ]
import json
with open('./extraxted_yaskawa.json','w' , encoding="utf-8") as f:
    f.write(json.dumps(extraxted , ensure_ascii=False  , indent=3))

In [10]:
import json
with open('./extraxted_yaskawa.json','r' , encoding="utf-8") as f:
    extraxted = json.load(f)

In [15]:

promptMerge = ChatPromptTemplate.from_messages([
    (
    "system",
    """
    You are a knowledge aggregation and normalization system.

    You will receive multiple JSON outputs extracted from different document chunks.

    Your task is to merge them into a single, non-redundant knowledge representation.

    ### Your goals

    1. Main Topics
    - Merge all main topics across chunks.
    - Remove duplicates.
    - Merge semantically similar topics into a single canonical topic.
    - If multiple topics refer to the same concept using different wording, keep only one.
    - Prefer broader high-level topics over fragmented ones.
    - Maximum 10 topics.

    2. Subtopics
    - Merge all subtopics.
    - Remove duplicates, paraphrases, and near-duplicates.
    - Combine highly related concepts into one canonical subtopic whenever possible.
    - If two subtopics have more than ~80% semantic overlap, keep only one.
    - Prefer the most common and descriptive name.
    - Maximum 30 subtopics.

    3. Summary
    - Read every chunk summary.
    - Produce ONE unified summary that represents the entire document.
    - Synthesize the summaries instead of concatenating them.
    - Remove repeated information.
    - Merge overlapping ideas into a single statement.
    - Preserve important facts and technical details.
    - Do NOT introduce new information.
    - Maximum 10 sentences (or about 10 lines).
    
    ### Normalization Rules

    - Merge synonyms into one canonical term.
    - Merge abbreviations with their full names.
    - Merge singular/plural variations.
    - Merge different spellings of the same concept.
    - Merge parent-child concepts if they essentially represent the same idea.
    - Never keep two items that express almost the same meaning.
    - Prefer concise, canonical names.
    - Do NOT invent new information.
    - Do NOT remove rare technical terms, protocol names, standards, version names, or error codes.
    - Favor precision over quantity.
    - The final output should contain as little redundancy as possible.

    ### Output Schema (JSON only)

    "main_topics": [],
    "subtopics": [],
    "sammary":''

    """
    ),
    (
        "user",
        """
        ### Input data:

        {chunks}
        """
    ),
])

chain  = promptMerge | llmExtranct

extraxtedMerge = []

MERGE_SIZE = 20
for i in range(0, len(extraxted), MERGE_SIZE ):
    extraxtedMerge.append( chain.invoke({ 'chunks':extraxted[i:i+MERGE_SIZE] } ).model_dump_json() )
    print(f'merge {i} / {len(extraxted)}')



merge 0 / 225
merge 20 / 225
merge 40 / 225
merge 60 / 225
merge 80 / 225
merge 100 / 225
merge 120 / 225
merge 140 / 225
merge 160 / 225
merge 180 / 225
merge 200 / 225
merge 220 / 225


In [16]:
print( extraxtedMerge[2])

{"main_topics":["Permanent Magnet Motor Auto-Tuning","PG Encoder Configuration and Tuning","Elevator Drive Parameter Configuration","Rescue Operation and Backup Power Systems","Drive Troubleshooting and Maintenance","Parameter Management and Software Tools"],"subtopics":["Auto-Tuning Types (Stationary, Rotational, Initial Pole Search)","Motor Nameplate Data Requirements (Power, Voltage, Current, Poles)","Encoder Offset and Z-Pulse Adjustment","PG-E3 Encoder Characteristics and Resolution","Safety Warnings (Sudden Movement, Electrical Shock, Uncoupling)","Mechanical Uncoupling and Brake Release Requirements","Digital I/O and Terminal Configuration (H1-H4)","Speed Reference and Multi-Speed Logic (d1-18, b1-01)","Torque Compensation and Load Cell Processing","Brake Sequence Control and Time Zones (t1-t9)","Acceleration/Deceleration Ramps and Jerk Adjustment (C1-C2)","Inertia Compensation and Riding Comfort Tuning (S3, C5)","Rescue Operation Configuration (S4-06 to S4-13)","UPS and Battery

In [17]:
promptProfile = ChatPromptTemplate.from_messages([
        (
        "system",
        """
You are an expert knowledge base profiling system.

You will receive aggregated metadata extracted from an entire document collection.

The input contains:
- Main topics
- Subtopics
- A synthesized collection summary

Your task is to generate a high-quality semantic profile of this knowledge base for an AI agent that must decide whether this RAG system is relevant for answering a user's question.

The profile should describe:

1. The primary domain and subject areas covered.
2. The overall scope and boundaries of the knowledge base.
3. The kinds of information users can retrieve.
4. The types of questions this knowledge base is well suited to answer.
5. The technical depth (overview, implementation, troubleshooting, reference, architecture, APIs, configuration, best practices, specifications, etc.) only when supported by the input.
6. The terminology, technologies, standards, protocols, products, or concepts that are well represented.
7. Any notable limitations or exclusions that can be inferred from the input.

The description should help an AI system decide whether this RAG should be queried before retrieval.

Requirements

- Produce ONE coherent paragraph between 120 and 220 words.
- Write in neutral, factual language.
- Synthesize the information instead of listing topics.
- Do not enumerate every topic or subtopic.
- Avoid redundancy.
- Do not invent information.
- Do not mention metadata, summaries, chunks, JSON, or extraction.
- Do not mention the existence of a RAG pipeline or document processing.
- Write as if this were the official description of the knowledge base itself.
- Emphasize retrieval capabilities rather than document structure.

The output should answer the implicit question:

"If a user asks something, what kinds of questions is this knowledge base capable of answering, and what knowledge does it contain?"

Return only the generated description.

        """
    ),
    (
        "user",
        """
        ### Input data:
        {merged_metadata}
        """
    )
])

from langchain_core.output_parsers import StrOutputParser

chainProfile = promptProfile | llm | StrOutputParser()

finalDes = chainProfile.invoke({'merged_metadata':extraxtedMerge }) 

print(finalDes)

This knowledge base comprises comprehensive technical documentation for the Yaskawa L1000A variable frequency drive series, specifically addressing CIMR-LB and CIMR-LT models across 200V and 400V classes. It provides detailed guidance on mechanical installation, electrical wiring, and safety compliance, including strict protocols for grounding, capacitor discharge, and hazardous energy isolation. The collection extensively covers drive configuration, detailing parameter setup for V/f, open-loop vector, closed-loop vector, and permanent magnet motor control modes. Advanced operational features such as auto-tuning, encoder configuration, and elevator-specific logic—including rescue operations, leveling, and anti-rollback control—are thoroughly documented. Users can retrieve information on troubleshooting fault codes, managing maintenance schedules for components like cooling fans and capacitors, and configuring communication protocols such as MEMOBUS/Modbus and CANopen. The content suppo